# ML model Hyperparameter Tuning

### Loading the CSV dataset in the Unity Catalog Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS ml_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/ml_lab"

# List of files to download
files = ["diabetes.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksUdemyCourse/refs/heads/main/MachineLearningModel/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Splitting the dataset into train and test values

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
   
data = spark.read.format("csv").option("header", "true").load(f"/Volumes/{catalog_name}/default/ml_lab/diabetes.csv")
data = data.dropna().select(col("Pregnancies").astype("int"),
                           col("Glucose").astype("int"),
                          col("BloodPressure").astype("int"),
                          col("SkinThickness").astype("int"),
                          col("Insulin").astype("int"),
                          col("BMI").astype("float"),
                          col("DiabetesPedigreeFunction").astype("float"),
                          col("Age").astype("int"),
                          col("Outcome").astype("int")
                          )

   
splits = data.randomSplit([0.7, 0.3])
train = splits[0]
test = splits[1]
print ("Training Rows:", train.count(), " Testing Rows:", test.count())

### Optimizing Hyperparameter values for our ML model

In [0]:
import mlflow
from mlflow.models import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, MinMaxScaler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import optuna
   
def objective(trial):
    # Suggest hyperparameter values using Optuna
    max_depth = trial.suggest_int('MaxDepth', 1, 10)
    max_bins = trial.suggest_categorical('MaxBins', [10, 20, 30])
    
    with mlflow.start_run(nested=True):
        # Train a model using the provided hyperparameter value
        numFeatures = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
        numVector = VectorAssembler(inputCols=numFeatures, outputCol="numericFeatures")
        numScaler = MinMaxScaler(inputCol=numVector.getOutputCol(), outputCol="normalizedFeatures")
        featureVector = VectorAssembler(inputCols=["normalizedFeatures"], outputCol="Features")
        mlAlgo = DecisionTreeClassifier(labelCol="Outcome",    
                                        featuresCol="Features",
                                        maxDepth=max_depth, maxBins=max_bins)
        pipeline = Pipeline(stages=[numVector, numScaler, featureVector, mlAlgo])
        model = pipeline.fit(train)
       
        # Evaluate the model to get the target metric
        prediction = model.transform(test)
        eval = MulticlassClassificationEvaluator(labelCol="Outcome", predictionCol="prediction", metricName="accuracy")
        accuracy = eval.evaluate(prediction)
        
        # Log parameters and metrics
        mlflow.log_param('MaxDepth', max_depth)
        mlflow.log_param('MaxBins', max_bins)
        mlflow.log_metric('accuracy', accuracy)
        
        # Infer and log model signature
        signature = infer_signature(train.select(numFeatures).toPandas(), prediction.select("prediction").toPandas())
        mlflow.spark.log_model(model, "model", signature=signature, dfs_tmpdir=f"/Volumes/{catalog_name}/default/ml_lab/tmp")
       
    # Optuna tries to maximize the objective, so return accuracy directly
    return accuracy

### Defining the Search Space for our hyperparameters

#### Also will be logging each hyperparameter run using a Trials() object

In [0]:
import optuna
from optuna.integration.mlflow import MLflowCallback

# Create an Optuna study to maximize accuracy
with mlflow.start_run(run_name='optuna_hyperparameter_tuning'):
    study = optuna.create_study(direction='maximize', study_name='diabetes_classification')
    
    # Run optimization
    study.optimize(objective, n_trials=3)
    
    # Get best parameters
    print("Best param values: ", study.best_params)
    print("Best accuracy: ", study.best_value)
    
    # Log best parameters to parent run
    mlflow.log_params(study.best_params)
    mlflow.log_metric('best_accuracy', study.best_value)
    
    # Display all trials
    print("\nAll trials:")
    for trial in study.trials:
        print(f"\nTrial {trial.number}: params={trial.params}, accuracy={trial.value}")